In [345]:
from openpyxl import load_workbook
import pandas as pd
from collections import defaultdict

In [346]:
def expandir_mescladas(ws):
    """
    Copia o valor da célula superior esquerda para todas as células
    pertencentes à região mesclada.
    """
    for merged in list(ws.merged_cells.ranges):

        valor = ws.cell(
            merged.min_row,
            merged.min_col
        ).value

        min_row = merged.min_row
        max_row = merged.max_row
        min_col = merged.min_col
        max_col = merged.max_col

        ws.unmerge_cells(str(merged))

        for r in range(min_row, max_row + 1):
            for c in range(min_col, max_col + 1):
                ws.cell(r, c).value = valor

In [347]:
def montar_cabecalho(ws):
    """
    Monta automaticamente o cabeçalho da planilha PPI.

    Expande células mescladas e propaga os títulos horizontal e
    verticalmente antes de concatenar os níveis do cabeçalho.
    """

    # Expande as células mescladas
    expandir_mescladas(ws)

    # Linhas do cabeçalho (Excel)
    HEADER_ROWS = [2, 3, 4, 5, 6, 7]

    # ------------------------------------------------------------------
    # Cria uma matriz contendo somente o cabeçalho
    # ------------------------------------------------------------------

    cab = []

    for r in HEADER_ROWS:

        linha = []

        for c in range(1, ws.max_column + 1):

            linha.append(ws.cell(r, c).value)

        cab.append(linha)

    cab = pd.DataFrame(cab)

    # ------------------------------------------------------------------
    # Propaga valores das células mescladas
    # ------------------------------------------------------------------

    # esquerda -> direita
    cab = cab.ffill(axis=1)

    # cima -> baixo
    cab = cab.ffill(axis=0)

    # ------------------------------------------------------------------
    # Limpa valores inúteis
    # ------------------------------------------------------------------

    lixo = {
        "#REF!",
        "TOTAL",
        "SOMA",
        0,
        "0",
        None,
        ""
    }

    cab = cab.replace(list(lixo), pd.NA)

    # ------------------------------------------------------------------
    # Monta o nome de cada coluna
    # ------------------------------------------------------------------

    colunas = []

    for c in range(cab.shape[1]):

        partes = []

        for r in range(cab.shape[0]):

            valor = cab.iat[r, c]

            if pd.isna(valor):
                continue

            valor = str(valor).strip()

            if valor == "":
                continue

            # Ignora fórmulas
            if valor.startswith("="):
                continue

            # Ignora erros do Excel
            if valor.startswith("#"):
                continue

            # Ignora TOTAL e SOMA
            if valor.upper() in ("TOTAL", "SOMA"):
                continue

            # Ignora números puros
            try:
                float(valor.replace(",", "."))
                continue
            except:
                pass

            # =============================================================
            # ✅ CORREÇÃO: aqui é onde o valor válido é adicionado à lista
            # =============================================================
            partes.append(valor)

        colunas.append(" - ".join(partes))

    # ------------------------------------------------------------------
    # Remove duplicados
    # ------------------------------------------------------------------

    contador = defaultdict(int)

    novas = []

    for nome in colunas:

        contador[nome] += 1

        if contador[nome] == 1:

            novas.append(nome)

        else:

            novas.append(f"{nome}__{contador[nome]}")

    return novas

In [348]:
ARQUIVO = "Planilha de Monitoramento_PPI_2025_31_03_25_PPI_R21 - 2025.xlsx"

wb = load_workbook(
    ARQUIVO,
    data_only=False
)

ws = wb["META(2025)"]

colunas = montar_cabecalho(ws)

c:\Users\carlos.ramos\OneDrive - Presidência da República\Área de Trabalho\Planilhas\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [349]:
dados = list(ws.values)

df = pd.DataFrame(
    dados[7:],   # linha 8 do Excel = índice 7
    columns=colunas
)

In [350]:
for i, c in enumerate(colunas):

    if "PAVIMENTAÇÃO" in c.upper():
        print(i, c)

9 DADOS CONTRATUAIS - PAVIMENTAÇÃO - Até 2022 - FÍSICO (km) - km (i) - EPL
10 DADOS CONTRATUAIS - PAVIMENTAÇÃO - Até 2022 - FÍSICO (km) - km (f) - EPL
11 DADOS CONTRATUAIS - PAVIMENTAÇÃO - Até 2022 - FÍSICO (km) - Ext. (km) - EPL
12 DADOS CONTRATUAIS - PAVIMENTAÇÃO - Até 2022 - PERCENTUAL (%) - Ext. (km) - EPL
13 DADOS CONTRATUAIS - PAVIMENTAÇÃO - Até 2022 - FINANCEIRO (R$) - Ext. (km) - EPL
14 DADOS CONTRATUAIS - PAVIMENTAÇÃO - FÍSICO (km) - km (i) - EPL
15 DADOS CONTRATUAIS - PAVIMENTAÇÃO - FÍSICO (km) - km (f) - EPL
16 DADOS CONTRATUAIS - PAVIMENTAÇÃO - FÍSICO (km) - Ext. (km) - EPL
17 DADOS CONTRATUAIS - PAVIMENTAÇÃO - PERCENTUAL (%) - Ext. (km) - EPL
18 DADOS CONTRATUAIS - PAVIMENTAÇÃO - FINANCEIRO (R$) - Ext. (km) - EPL
19 DADOS CONTRATUAIS - PAVIMENTAÇÃO - FÍSICO (km) - km (i) - EPL__2
20 DADOS CONTRATUAIS - PAVIMENTAÇÃO - FÍSICO (km) - km (f) - EPL__2
21 DADOS CONTRATUAIS - PAVIMENTAÇÃO - FÍSICO (km) - Ext. (km) - EPL__2
22 DADOS CONTRATUAIS - PAVIMENTAÇÃO - PERCENTUAL (%) - Ex

In [351]:
for i, c in enumerate(colunas):

    if "PROCESSO" in c.upper():
        print(i, c)

191 METAS ANO PRESENTE - RISCOS (Pavimentação) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
211 METAS ANO PRESENTE - RISCOS (Duplicação) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
231 METAS ANO PRESENTE - RISCOS (OAE) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
251 METAS ANO PRESENTE - RISCOS (Contorno) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
271 METAS ANO PRESENTE - RISCOS (FX Adicional) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
291 METAS ANO PRESENTE - RISCOS (Terceira Faixa) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)


In [352]:
indice_colunas = pd.DataFrame({
    "indice": range(len(df.columns)),
    "coluna": df.columns
})

display(indice_colunas)

,indice,coluna
0,0,Região - Região - Região - Região - Norte
1,1,SETOR - SETOR - SETOR - SETOR - Rodoviário
2,2,UF - UF - UF - UF - Rodoviário
3,3,ESTADO/LOTE - ESTADO/LOTE - Nº Concessões: - Nº Concessões: - RIO GRANDE DO SUL
4,4,BR - BR
...,...,...
579,579,"METAS ANO PRESENTE - OBSERVAÇÕES - SITUAÇÃO GERAL - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta) - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta)__283"
580,580,"METAS ANO PRESENTE - OBSERVAÇÕES - SITUAÇÃO GERAL - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta) - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta)__284"
581,581,"METAS ANO PRESENTE - OBSERVAÇÕES - SITUAÇÃO GERAL - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta) - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta)__285"
582,582,"METAS ANO PRESENTE - OBSERVAÇÕES - SITUAÇÃO GERAL - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta) - (Quando escolher a opção Outros, descrever aqui a situação de forma suscinta)__286"


In [353]:
indice_colunas[
    indice_colunas["coluna"].str.contains(
        "PROCESSO",
        case=False,
        na=False
    )
]

,indice,coluna
191,191,METAS ANO PRESENTE - RISCOS (Pavimentação) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
211,211,METAS ANO PRESENTE - RISCOS (Duplicação) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
231,231,METAS ANO PRESENTE - RISCOS (OAE) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
251,251,METAS ANO PRESENTE - RISCOS (Contorno) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
271,271,METAS ANO PRESENTE - RISCOS (FX Adicional) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)
291,291,METAS ANO PRESENTE - RISCOS (Terceira Faixa) - Nº PROCESSO - (caso o processo já tenha sido protocolado) - (caso o processo já tenha sido protocolado)


In [354]:
indice_colunas[
    indice_colunas["coluna"].str.contains(
        "LICEN",
        case=False,
        na=False
    )
]

,indice,coluna
193,193,METAS ANO PRESENTE - RISCOS (Pavimentação) - LICENÇA - Foi solicitado licença para o segmento? - Foi solicitado licença para o segmento?
194,194,METAS ANO PRESENTE - RISCOS (Pavimentação) - LICENÇA - O pedido de licença desse segmento está dentro de um pedido de trecho maior? - O pedido de licença desse segmento está dentro de um pedido de trecho maior?
195,195,"METAS ANO PRESENTE - RISCOS (Pavimentação) - LICENÇA - Se positivo, especificar o trecho completo requisitado - km(i) - km(f). - Se positivo, especificar o trecho completo requisitado - km(i) - km(f)."
213,213,METAS ANO PRESENTE - RISCOS (Duplicação) - LICENÇA - Foi solicitado licença para o segmento? - Foi solicitado licença para o segmento?
214,214,METAS ANO PRESENTE - RISCOS (Duplicação) - LICENÇA - O pedido de licença desse segmento está dentro de um pedido de trecho maior? - O pedido de licença desse segmento está dentro de um pedido de trecho maior?
215,215,"METAS ANO PRESENTE - RISCOS (Duplicação) - LICENÇA - Se positivo, especificar o trecho completo requisitado - km(i) - km(f). - Se positivo, especificar o trecho completo requisitado - km(i) - km(f)."
233,233,METAS ANO PRESENTE - RISCOS (OAE) - LICENÇA - Foi solicitado licença para o segmento? - Foi solicitado licença para o segmento?
234,234,METAS ANO PRESENTE - RISCOS (OAE) - LICENÇA - O pedido de licença desse segmento está dentro de um pedido de trecho maior? - O pedido de licença desse segmento está dentro de um pedido de trecho maior?
235,235,"METAS ANO PRESENTE - RISCOS (OAE) - LICENÇA - Se positivo, especificar o trecho completo requisitado - km(i) - km(f). - Se positivo, especificar o trecho completo requisitado - km(i) - km(f)."
253,253,METAS ANO PRESENTE - RISCOS (Contorno) - LICENÇA - Foi solicitado licença para o segmento? - Foi solicitado licença para o segmento?


In [355]:
indice_colunas[
    indice_colunas["coluna"].str.contains(
        "FINANCEIRO PLAN",
        case=False,
        na=False
    )
]

,indice,coluna
183,183,METAS ANO PRESENTE - PAVIMENTAÇÃO (Ano Vigente) - FINANCEIRO PLAN (R$) - Ext. (km)
203,203,"METAS ANO PRESENTE - DUPLICAÇÃO (Ano Vigente) - FINANCEIRO PLAN (R$) - (Colocar aqui observações gerais; Quando escolher a opção Outros, descrever a situação de forma suscinta.)"
223,223,"METAS ANO PRESENTE - OAE (Ano Vigente) - FINANCEIRO PLAN (R$) - (Colocar aqui observações gerais; Quando escolher a opção Outros, descrever a situação de forma suscinta.)"
243,243,"METAS ANO PRESENTE - CONTORNO (Ano Vigente) - FINANCEIRO PLAN (R$) - (Colocar aqui observações gerais; Quando escolher a opção Outros, descrever a situação de forma suscinta.)"
263,263,"METAS ANO PRESENTE - FX ADICIONAL (Ano Vigente) - FINANCEIRO PLAN (R$) - (Colocar aqui observações gerais; Quando escolher a opção Outros, descrever a situação de forma suscinta.)"
283,283,"METAS ANO PRESENTE - TERCEIRA FAIXA (Ano Vigente) - FINANCEIRO PLAN (R$) - (Colocar aqui observações gerais; Quando escolher a opção Outros, descrever a situação de forma suscinta.)"


In [356]:
df.columns = [
    str(c)
        .replace("\n", " ")
        .replace("\r", " ")
        .strip()
    for c in df.columns
]

In [357]:
print("Total de colunas:", len(df.columns))
print("Colunas únicas:", len(set(df.columns)))

Total de colunas: 584
Colunas únicas: 584


In [358]:
import pandas as pd

cab = []

for c in range(1, ws.max_column + 1):

    info = {
        "coluna": c
    }

    for r in range(1, 9):

        info[f"L{r}"] = ws.cell(r, c).value

    cab.append(info)

cab = pd.DataFrame(cab)

cab.head()

,coluna,L1,L2,L3,L4,L5,L6,L7,L8
0,1,NaN,NaN,Região,None,NaN,NaN,Norte,Norte
1,2,NaN,NaN,SETOR,None,NaN,NaN,Rodoviário,Rodoviário
2,3,NaN,,UF,None,NaN,NaN,None,RS
3,4,PLANILHA DE MONITORAMENTO,NaN,ESTADO/LOTE,None,Nº Concessões:,NaN,RIO GRANDE DO SUL,RIO GRANDE DO SUL
4,5,NaN,NaN,BR,None,=E7+E30+E43+E51+E78+E91+E115+E169+E172+E175+E177+E184+E194+E201+E208+E212+E216+E230+E237+E244+E248+E252+E259+E265+E269+E291+E328+E332+E336+E343+E347+E354+E379+E383,NaN,1,101


In [359]:
def limpar(x):

    if x is None:
        return ""

    x = str(x).strip()

    if x.startswith("="):
        return ""

    if x == "#REF!":
        return ""

    return x

for c in cab.columns[1:]:

    cab[c] = cab[c].apply(limpar)

In [360]:
pd.set_option("display.max_colwidth", None)

cab.iloc[175:205]

,coluna,L1,L2,L3,L4,L5,L6,L7,L8
175,176,PER - PROGRAMA DE EXPLORAÇÃO DA RODOVIA,nan,TERCEIRA FAIXA,PÓS 2026,PERCENTUAL (%),nan,,
176,177,PER - PROGRAMA DE EXPLORAÇÃO DA RODOVIA,nan,TERCEIRA FAIXA,PÓS 2026,FINANCEIRO (R$),nan,,
177,178,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),FÍSICO (km),Descrição,nan,,
178,179,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),FÍSICO (km),km (i),nan,,
179,180,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),FÍSICO (km),km (f),nan,,
180,181,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),FÍSICO (km),Ext. (km),nan,,
181,182,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),PERCENTUAL (%) PLAN,PAV. TOTAL PLAN,nan,TOTAL,SOMA
182,183,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),(km)% PLAN,,nan,,
183,184,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),FINANCEIRO PLAN (R$),,nan,,
184,185,META ANO 2025,METAS ANO PRESENTE,PAVIMENTAÇÃO (Ano Vigente),PERCENTUAL (%) EXEC,PAV. TOTAL EXEC,nan,TOTAL,SOMA


In [361]:
# %% =====================================================================
# CRIAÇÃO DO DATAFRAME FINAL COM AS 43 COLUNAS ESPECIFICADAS
# ========================================================================

# Lista atualizada com as novas colunas
colunas_alvo = [
    "ID-ÚNICO",
    "SETOR",
    "UF",
    "BR",
    "EMPREENDIMENTO",
    "EXECUTOR (Grupo Controlador)",
    "DATA DE INÍCIO DA CONCESSÃO",
    "ANO DA CONCESSÃO",
    "INTERFERÊNCIAS",
    "INTERVENÇÃO AFETADA",
    "Descrição",
    "km(i)",
    "km(f)",
    "Ext.(km)",
    "Ext.(km)!",
    "FINANCEIRO(R$)",
    "FINANCEIRO(R$)!",
    "PER ANO(Execução)",
    "LP",
    "LI",
    "LO",
    "DATA DE INÍCIO",
    "SINAFLOR",
    "SISGLAF",
    "SEI",
    "Risco Principal",
    "CRITICIDADE",
    "SITUAÇÃO",
    "COMENTÁRIOS GERAIS",
    "Data do Atendimento",
    "NÍVEL DE ALERTA",
    "Emitida",
    "Em análise",
    "Aguardando",
    "Solicitada",
    "Sem informação",
    "Pendência",
    "Outras Descrições",
    # ===== NOVAS COLUNAS ADICIONADAS =====
    "Nº PROCESSO",
    "DATA DO PROTOCOLO",
    "LICENÇA",
    "SITUAÇÃO GERAL",
    # ====================================
    "PLANILHA"
]

# Cria o DataFrame final vazio, com o mesmo índice do df original
df_final = pd.DataFrame(index=df.index, columns=colunas_alvo)

# ========================================================================
# 1. DEFINIÇÃO DOS MAPEAMENTOS (palavras-chave para cada coluna destino)
# ========================================================================

mapeamento = {
    "ID-ÚNICO": ["ID", "UNICO"],
    "SETOR": ["SETOR"],
    "UF": ["UF"],
    "BR": ["BR"],
    "EMPREENDIMENTO": ["EMPREENDIMENTO"],
    "EXECUTOR (Grupo Controlador)": ["EXECUTOR", "CONTROLADOR"],
    "DATA DE INÍCIO DA CONCESSÃO": ["CONCESSÃO", "INÍCIO"],
    "ANO DA CONCESSÃO": ["ANO", "CONCESSÃO"],
    "INTERFERÊNCIAS": ["INTERFER"],
    "INTERVENÇÃO AFETADA": ["INTERVENÇÃO", "AFETADA"],
    "Descrição": ["DESCRIÇÃO"],
    "km(i)": ["KM I", "KM INICIAL"],
    "km(f)": ["KM F", "KM FINAL"],
    "Ext.(km)": ["EXT", "KM"],  # pega a primeira com EXT e KM
    "Ext.(km)!": ["EXT", "!", "KM"],  # pega a que tem "!"
    "FINANCEIRO(R$)": ["FINANCEIRO", "R$"],
    "FINANCEIRO(R$)!": ["FINANCEIRO", "R$", "!"],
    "PER ANO(Execução)": ["PER ANO", "EXECUÇÃO"],
    "LP": ["LP"],
    "LI": ["LI"],
    "LO": ["LO"],
    "DATA DE INÍCIO": ["DATA DE INÍCIO"],  # será tratado com exclusão de "CONCESSÃO"
    "SINAFLOR": ["SINAFLOR"],
    "SISGLAF": ["SISGLAF"],
    "SEI": ["SEI"],
    "Risco Principal": ["RISCO", "PRINCIPAL"],
    "CRITICIDADE": ["CRITICIDADE"],
    "SITUAÇÃO": ["SITUAÇÃO"],  # sem a palavra GERAL para não confundir
    "COMENTÁRIOS GERAIS": ["COMENTÁRIOS", "GERAIS"],
    "Data do Atendimento": ["ATENDIMENTO"],
    "NÍVEL DE ALERTA": ["ALERTA"],
    "Emitida": ["EMITIDA"],
    "Em análise": ["ANÁLISE"],
    "Aguardando": ["AGUARDANDO"],
    "Solicitada": ["SOLICITADA"],
    "Sem informação": ["SEM INFORMAÇÃO"],
    "Pendência": ["PENDÊNCIA"],
    "Outras Descrições": ["OUTRAS", "DESCRI"],
    # ===== MAPEAMENTO DAS NOVAS COLUNAS =====
    "Nº PROCESSO": ["PROCESSO", "Nº"],  # ou "NÚMERO" se preferir
    "DATA DO PROTOCOLO": ["PROTOCOLO", "DATA"],
    "LICENÇA": ["LICENÇA"],
    "SITUAÇÃO GERAL": ["SITUAÇÃO", "GERAL"],  # distinta de "SITUAÇÃO" sozinha
    # ========================================
    "PLANILHA": []  # será preenchida manualmente
}

# ========================================================================
# 2. FUNÇÃO PARA ENCONTRAR A COLUNA DE ORIGEM
# ========================================================================

def encontrar_coluna_origem(palavras, ignorar=None):
    """
    Retorna o nome da coluna em df.columns que contém TODAS as palavras da lista.
    Se 'ignorar' for passado, ignora colunas que contenham essa palavra.
    """
    for col in df.columns:
        col_str = str(col).upper()
        # Se tiver palavra para ignorar, pula
        if ignorar and ignorar.upper() in col_str:
            continue
        # Verifica se todas as palavras estão presentes
        if all(p.upper() in col_str for p in palavras):
            return col
    return None

# ========================================================================
# 3. PREENCHIMENTO COLUNA POR COLUNA
# ========================================================================

print("🔍 Mapeamento realizado:\n")

for destino, palavras in mapeamento.items():
    
    # Caso especial: PLANILHA
    if destino == "PLANILHA":
        df_final[destino] = "META (2025)"
        print(f"  {destino:30} <- (preenchido com 'META (2025)')")
        continue
    
    # Caso especial: "DATA DE INÍCIO" (da intervenção) deve ignorar "CONCESSÃO"
    if destino == "DATA DE INÍCIO":
        origem = encontrar_coluna_origem(palavras, ignorar="CONCESSÃO")
    else:
        origem = encontrar_coluna_origem(palavras)
    
    if origem:
        df_final[destino] = df[origem]
        print(f"  {destino:30} <- {origem}")
    else:
        print(f"  {destino:30} <- ⚠️ NÃO ENCONTRADO (deixado como NaN)")

# ========================================================================
# 4. (OPCIONAL) AJUSTES FINOS MANUAIS
# ========================================================================
# Se alguma coluna foi mapeada errada, você pode sobrescrever aqui.
# Exemplo:
# col_correta = encontrar_coluna_origem(["PROCESSO"], ignorar="Nº")  # ajuste conforme necessário
# if col_correta:
#     df_final["Nº PROCESSO"] = df[col_correta]

# ========================================================================
# 5. EXIBE E SALVA
# ========================================================================

print("\n📊 Primeiras linhas do DataFrame final:")
print(df_final.head())

print(f"\nFormato final: {df_final.shape[0]} linhas x {df_final.shape[1]} colunas")

# Salva em Excel
arquivo_final = "PPI_Colunas_Especificas.xlsx"
with pd.ExcelWriter(arquivo_final, engine="openpyxl") as writer:
    df_final.to_excel(writer, index=False, sheet_name="META_2025")

print(f"\n✅ Arquivo exportado: {arquivo_final}")

🔍 Mapeamento realizado:

  ID-ÚNICO                       <- ⚠️ NÃO ENCONTRADO (deixado como NaN)
  SETOR                          <- SETOR - SETOR - SETOR - SETOR - Rodoviário
  UF                             <- UF - UF - UF - UF - Rodoviário
  BR                             <- BR - BR
  EMPREENDIMENTO                 <- DADOS CONTRATUAIS - EMPREENDIMENTO - EMPREENDIMENTO - CONCESSÃO VIA SUL
  EXECUTOR (Grupo Controlador)   <- DADOS CONTRATUAIS - EXECUTOR             (Grupo Controlador) - EXECUTOR             (Grupo Controlador) - MOTIVA
  DATA DE INÍCIO DA CONCESSÃO    <- ⚠️ NÃO ENCONTRADO (deixado como NaN)
  ANO DA CONCESSÃO               <- ⚠️ NÃO ENCONTRADO (deixado como NaN)
  INTERFERÊNCIAS                 <- METAS ANO PRESENTE - RISCOS (Pavimentação) - INTERFERÊNCIA - (células contêm listas suspensas) - (células contêm listas suspensas)
  INTERVENÇÃO AFETADA            <- ⚠️ NÃO ENCONTRADO (deixado como NaN)
  Descrição                      <- METAS ANO PRESENTE - PAVIMENTAÇÃO

In [362]:
arquivo_saida = "PPI_Tratada.xlsx"

with pd.ExcelWriter(
    arquivo_saida,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        index=False,
        sheet_name="META(2025)"
    )

print("Exportação concluída.")

Exportação concluída.
